### 生成数据类型为数组[]的ROOT文件branch

一个事件可能击中40个塑闪中其中几个，一个事件在某一个塑闪上可能会有多重击中

*Br    0 :eventID   : eventID/I                                              *


*Br    1 :nHit      : nHit/I                                                 *

*Br    2 :PlasticID : PlasticID[nHit]/I                                      *

*Br    3 :TURaw     : TURaw[nHit]/D                                          *


*Br    4 :fTURaw    : fTURaw[40]/D                                           *


In [2]:
TFile *fout = new TFile("hod_hit.root", "RECREATE");
TTree *tree = new TTree("tree", "HOD hit tree with PlasticID and nHit");

const Int_t NPlastic = 40;
const Int_t MaxHit   = 3;

Int_t eventID;
Int_t nHit;

Int_t PlasticID[MaxHit];     // 当前 event 中每个 hit 对应的塑闪 ID
Double_t TURaw[MaxHit];      // 每个 hit 对应的 TDC 值

Double_t fTURaw[NPlastic];   // 固定数组：40 个塑闪完整状态

tree->Branch("eventID", &eventID, "eventID/I");
tree->Branch("nHit", &nHit, "nHit/I");

tree->Branch("PlasticID", PlasticID, "PlasticID[nHit]/I");
tree->Branch("TURaw", TURaw, "TURaw[nHit]/D");
tree->Branch("fTURaw", fTURaw, "fTURaw[40]/D");

TRandom3 rand(0);

for(int iev = 0; iev < 50000; iev++) //外层事件循环
{
    eventID = iev;

    for(int i = 0; i < NPlastic; i++) // 初始化 40 个塑闪
    {
        fTURaw[i] = -999;
    }

    for(int i = 0; i < MaxHit; i++) // 初始化 hit 数组
    {
        PlasticID[i] = -1;
        TURaw[i] = -999;
    }

    // nHit = 0, 1, 2, 3
    nHit = rand.Integer(MaxHit + 1);

    for(int ihit = 0; ihit < nHit; ihit++)
    {
        // 允许同一个 event 中重复击中同一个 PlasticID
        int plasticID = rand.Integer(NPlastic) + 1;  // 1 到 40
        double tdc = rand.Gaus(100, 10);

        PlasticID[ihit] = plasticID;//把变量 plasticID 的值，放进 PlasticID 数组的第 ihit 个位置；
        TURaw[ihit] = tdc; //把变量 tdc 的值，放进 TURaw 数组的第 ihit 个位置。

        // 注意：如果 plasticID 重复，fTURaw[plasticID-1] 会被后一次 hit 覆盖
        fTURaw[plasticID - 1] = tdc;
    }

    tree->Fill();
}

tree->Write();
fout->Close();

std::cout << "Created file: hod_hit.root" << std::endl;

Created file: hod_hit.root


tree->Draw("PlasticID", "nHit==1");画的是单次击中时的塑闪ID分布

tree->Draw("PlasticID[1]");画的是每个event中第2次 hit的塑闪ID分布

PlasticID[0]  存第 1 次 hit 的塑闪 ID

PlasticID[1]  存第 2 次 hit 的塑闪 ID

PlasticID[2]  存第 3 次 hit 的塑闪 ID